In [1]:
!pip install unsloth numpy bitsandbytes datasets transformers trl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 

In [16]:
import torch
import numpy as np

# Step 1 - load_base_model_and_tokenizer
from unsloth import FastLanguageModel


In [3]:

def load_base_model_and_tokenizer(model_name='unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit', max_seq_length=256):
    """Load a 4-bit quantized causal LM and its tokenizer via Unsloth.

    Returns:
        (model, tokenizer)
    """
    # TODO: call FastLanguageModel.from_pretrained with 4-bit loading and return (model, tokenizer)

    model , tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        dtype = None,
        load_in_4bit = True, # 4 bit loading
    )

    return (model,tokenizer)


In [4]:
# Step 2 - count_total_parameters
def count_total_parameters(model):
    """Return the total number of parameters in `model` as a Python int."""
    # TODO: sum p.numel() over every parameter tensor in the module
    result = sum(p.numel() for p in model.parameters())
    return result


In [5]:

# Step 3 - is_model_4bit_quantized
from bitsandbytes.nn import Linear4bit

def is_model_4bit_quantized(model):
    """Return True if any submodule of `model` is a bitsandbytes 4-bit linear layer."""
    # TODO: walk the model's submodules and check for a bitsandbytes Linear4bit instance
    #sanity checking confirming that our unsloth loads actually gave us a 4-bit
    # quantized backbone
    for idx, module in enumerate(model.modules()):
        if(isinstance(module,Linear4bit)):
            return True

    return False

In [6]:
# Step 4 - ensure_pad_token
def ensure_pad_token(tokenizer):
    """Guarantee tokenizer.pad_token is not None; fall back to eos_token."""
    # TODO: if the tokenizer is missing a pad token, reuse its eos token
    # most decoder only tokens have pad_tokens missing, using eos_token id for padding is safe
    # if not tokenizer.pad_token:
    #     tokenizer.pad_token = tokenizer.eos_token
    #     # if an empty-string pad token is treated as alredy set
    #     if not tokenizer.pad_token:
    #         tokenizer.pad_token = '</s>'
    # return tokenizer
    if getattr(tokenizer,'pad_token',None) is None:
        tokenizer.pad_token = getattr(
            tokenizer,'eos_token','</s>'
        )

    return tokenizer


In [7]:

# Step 5 - get_lora_target_modules
def get_lora_target_modules():
    """Return the attention projection module name suffixes for LoRA."""
    # TODO: return the list of attention projection module names LoRA should adapt

    return ['q_proj','k_proj','v_proj','o_proj']


In [8]:
# Step 6 - attach_lora_adapters
def attach_lora_adapters(model, r=8, lora_alpha=16, target_modules=None):
    """Wrap the base model with LoRA adapters and return the PEFT model."""
    # TODO: wrap `model` with LoRA via FastLanguageModel.get_peft_model using r, lora_alpha, target_modules

    if target_modules is None:
        target_modules = ['q_proj','k_proj','v_proj','o_proj']

    peft_model = FastLanguageModel.get_peft_model(
        model,
        r=r,
        lora_alpha=lora_alpha,
        target_modules=target_modules
        )
    return peft_model


In [9]:

# Step 7 - count_trainable_parameters
def count_trainable_parameters(model):
    """Return the number of trainable parameters in `model`."""
    # TODO: sum p.numel() over model.parameters() where requires_grad is True
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad == True)
    return trainable_params

# Step 8 - trainable_fraction
def trainable_fraction(trainable_count, total_count):
    # TODO: return the fraction of parameters that are trainable.
    return trainable_count / total_count

# Step 9 - build_instruction_examples
def build_instruction_examples():
    """Return a small list of {'instruction', 'response'} dicts for SFT."""
    # TODO: return a tiny hand-written list of instruction/response example dicts.
    return [
        {
            "instruction":"What is the capital of India ?",
            "response":"India"
        },
        {
            "instruction":"What is the capital of China ?",
            "response":"Beijing",
        },
        {
            "instruction":"What is the capital of USA ?",
            "response":"Washington DC",
        }
    ]

# Step 10 - format_instruction_example
def format_instruction_example(example):
    """Return a single training string with role markers for instruction and response."""
    # TODO: combine example['instruction'] and example['response'] into one string
    return f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}"

# Step 11 - format_all_examples
def format_all_examples(examples):
    """Format each instruction/response dict into a training string."""
    # TODO: apply format_instruction_example to every example and return the list
    examples_list = []
    for example in examples:
        response = format_instruction_example(example)
        examples_list.append(response)
    return examples_list


In [10]:
# Step 12 - build_text_dataset
from datasets import Dataset

def build_text_dataset(texts):
    """Wrap a list of training strings in a HF Dataset with a 'text' column."""
    # TODO: return a datasets.Dataset with one 'text' column holding the given strings
    return Dataset.from_dict({'text': texts})

# Step 13 - tokenize_text
def tokenize_text(tokenizer, text):
    """Tokenize a single string and return a list[int] of input ids."""
    # TODO: call the tokenizer on text and return its input_ids as a plain list
    return tokenizer.encode(text)

# Step 14 - count_tokens
def count_tokens(input_ids):
    """Return the number of tokens in a tokenized example."""
    # TODO: return the length of the input_ids sequence
    return len(input_ids)



In [11]:
from transformers import TrainingArguments

def build_training_arguments(output_dir='./sft_out', max_steps=5, learning_rate=2e-4):
    """Return featherweight TrainingArguments for the SFT run."""
    # TODO: build TrainingArguments with batch size 1, given max_steps, given lr, bf16 or fp16.
    bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=learning_rate,
        logging_steps=1,
        optim='adamw_8bit',
        max_steps=max_steps,
        bf16=bf16_supported,
        fp16= not bf16_supported,
        per_device_train_batch_size=1
    )

    return training_args


In [12]:
from trl import SFTTrainer

def build_sft_trainer(model, tokenizer, dataset, training_args, max_seq_length=256):
    """Construct a trl SFTTrainer over dataset['text'] ready to .train()."""
    # TODO: wire model, tokenizer, dataset, and training_args into an SFTTrainer
    trainer = SFTTrainer(
        model=model,
        train_dataset = dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        args=training_args,
        packing=False,
    )
    return trainer

# Step 17 - run_sft_training
def run_sft_training(trainer):
    """Run a few SFT steps and return the final training loss as a float."""
    # TODO: drive the trainer through its short optimization run and return the final loss
    # output = trainer.train()
    # return output.training_loss
    # disable checkpoint saving to avoid picklingerror on sftconfig
    trainer.args.save_strategy = "no"

    #disable multiprocessing dataloading to avoid cross process serializzation
    trainer.args.dataloader_num_workers = 0

    output = trainer.train()

    return float(output.training_loss)


In [13]:
# Step 18 - switch_to_inference_mode
from unsloth import FastLanguageModel

def switch_to_inference_mode(model):
    """Switch the LoRA-tuned model into Unsloth's fast inference mode and return it."""
    # TODO: call the Unsloth helper that prepares the model for fast generation
    model = FastLanguageModel.for_inference(model)
    return model

# Step 19 - build_chat_prompt
def build_chat_prompt(tokenizer, instruction):
    """Return a chat-template prompt string ready for assistant generation."""
    # TODO: wrap the instruction as a user turn and produce the assistant-generation prompt string
    messages = [
        {"role":"user",
        "content":instruction}
    ]

    prompt_string = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return prompt_string

# Step 20 - generate_reply
def generate_reply(model, tokenizer, prompt, max_new_tokens=32):
    """Greedy-generate a reply for `prompt` and return the decoded text."""
    # TODO: tokenize prompt, run model.generate with do_sample=False, decode new tokens only

    # build input ids and move to the model's current device
    inputs = tokenizer(prompt,return_tensors='pt').to(model.device)

    #record the length of the prompts to slice it out later
    input_length = inputs.input_ids.shape[1]

    # decode using the model
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    # isolate just the newly generated tokens
    new_tokens = outputs[0, input_length:]

    # clean text
    return tokenizer.decode(new_tokens,skip_special_tokens=True)


In [14]:

def main():
    torch.manual_seed(0)

    # 1) Load 4-bit base model + tokenizer.
    model, tokenizer = load_base_model_and_tokenizer(
        model_name="unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
        max_seq_length=256,
    )
    total_params = count_total_parameters(model)
    quantized = is_model_4bit_quantized(model)
    print(f"[base] total_params={total_params:,} 4bit={quantized}")

    # 2) Make sure tokenizer has a pad token.
    tokenizer = ensure_pad_token(tokenizer)
    print(f"[tokenizer] pad_token={tokenizer.pad_token!r}")

    # 3) Attach LoRA adapters to attention projections.
    target_modules = get_lora_target_modules()
    print(f"[lora] target_modules={target_modules}")
    model = attach_lora_adapters(
        model, r=8, lora_alpha=16, target_modules=target_modules
    )
    trainable = count_trainable_parameters(model)
    frac = trainable_fraction(trainable, total_params)
    print(f"[lora] trainable={trainable:,} fraction={frac:.6f}")

    # 4) Build a tiny in-code instruction dataset.
    examples = build_instruction_examples()
    print(f"[data] num_examples={len(examples)}")
    first_text = format_instruction_example(examples[0])
    print(f"[data] formatted[0]={first_text!r}")
    texts = format_all_examples(examples)
    dataset = build_text_dataset(texts)
    print(f"[data] dataset_columns={dataset.column_names} size={len(dataset)}")

    # 5) Peek at tokenization of one example.
    ids = tokenize_text(tokenizer, texts[0])
    print(f"[data] tokens[0]={count_tokens(ids)}")

    # 6) Featherweight SFT: 5 steps, batch size 1.
    training_args = build_training_arguments(
        output_dir="./sft_out", max_steps=5, learning_rate=2e-4
    )
    trainer = build_sft_trainer(
        model, tokenizer, dataset, training_args, max_seq_length=256
    )
    final_loss = run_sft_training(trainer)
    print(f"[train] final_loss={final_loss:.4f}")

    # 7) Switch to fast inference and generate a reply.
    switch_to_inference_mode(model)
    prompt = build_chat_prompt(tokenizer, "Say hello in one short sentence.")
    reply = generate_reply(model, tokenizer, prompt, max_new_tokens=32)
    print(f"[gen] reply={reply!r}")

    passed = (
        total_params > 0
        and trainable > 0
        and trainable < total_params
        and 0.0 < frac < 0.1
        and isinstance(reply, str)
        and len(reply) > 0
        and final_loss == final_loss  # finite (not NaN)
    )
    print({"passed": bool(passed)})
    print("PASS" if passed else "FAIL")



In [17]:
main()

==((====))==  Unsloth 2026.7.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[base] total_params=315,119,488 4bit=True
[tokenizer] pad_token='<|vision_pad|>'
[lora] target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.7.6 patched 24 layers with 24 QKV layers, 24 O layers and 0 MLP layers.


[lora] trainable=1,081,344 fraction=0.003432
[data] num_examples=3
[data] formatted[0]='### Instruction:\nWhat is the capital of India ?\n\n### Response:\nIndia'
[data] dataset_columns=['text'] size=3
[data] tokens[0]=14


...

Step,Training Loss
1,4.221665
2,4.791457
3,4.418325
4,4.418325
5,3.767714


Both `max_new_tokens` (=32) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[train] final_loss=4.3235


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

[gen] reply="Hello! It's nice to meet you!"
{'passed': True}
PASS
